# 23.5 — Social choice & voting

Voting rules are mechanisms: they translate rankings into outcomes, and every translation creates incentives. Save a copy to Drive to edit.

Rankings describe each voter's preference order over candidates. Pairwise majority reuses game-theory deviation logic: one alternative beats another if more voters rank it higher, but the lesson's Condorcet cycle shows majority preference need not be transitive.

In [ ]:
import itertools
import math

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(23)

np.set_printoptions(precision=3, suppress=True)


## The concept, built once: from rankings to rule-specific winners

A voting rule maps a profile of rankings to an outcome. We first implement plurality and Borda exactly as in the lesson: plurality counts only first-place votes, while three-candidate Borda assigns $2,1,0$ points down each ranking.

In [ ]:
def plurality(profile, candidates, tie_break=None):
    counts = {candidate: 0 for candidate in candidates}
    for ballot in profile:
        counts[ballot[0]] += 1

    best = max(counts.values())
    winners = [candidate for candidate in candidates if counts[candidate] == best]
    if tie_break is None:
        winner = tuple(winners)
    else:
        order = {candidate: i for i, candidate in enumerate(tie_break)}
        winner = sorted(winners, key=lambda candidate: order[candidate])[0]

    return winner, counts


def borda(profile, candidates):
    m = len(candidates)
    scores = {candidate: 0 for candidate in candidates}
    for ballot in profile:
        for rank, candidate in enumerate(ballot):
            scores[candidate] += m - rank - 1

    best = max(scores.values())
    winners = tuple(candidate for candidate in candidates if scores[candidate] == best)
    return winners, scores


lesson_profile = [
    ("A", "B", "C"),
    ("A", "B", "C"),
    ("B", "C", "A"),
    ("C", "B", "A"),
    ("C", "A", "B"),
]
lesson_candidates = ("A", "B", "C")
plural_winner, plural_counts = plurality(lesson_profile, lesson_candidates)
borda_winner, borda_scores = borda(lesson_profile, lesson_candidates)

assert plural_counts == {"A": 2, "B": 1, "C": 2}
assert plural_winner == ("A", "C")
assert borda_scores == {"A": 5, "B": 6, "C": 4}
assert borda_winner == ("B",)

print("Plurality counts:", plural_counts)
print("Plurality winner set:", plural_winner)
print("Borda scores:", borda_scores)
print("Borda winner:", borda_winner)


## Pairwise majority and agenda dependence

For the Condorcet profile $(A,B,C),(B,C,A),(C,A,B)$, the lesson's exact margins are $A\succ B$ by $2$ to $1$, $B\succ C$ by $2$ to $1$, and $C\succ A$ by $2$ to $1$. An agenda rule therefore needs both a first pair and a tie-breaking convention.

In [ ]:
def pairwise_matrix(profile, candidates):
    n = len(candidates)
    index = {candidate: i for i, candidate in enumerate(candidates)}
    wins = np.zeros((n, n), dtype=int)

    for ballot in profile:
        rank = {candidate: i for i, candidate in enumerate(ballot)}
        for left in candidates:
            for right in candidates:
                if left == right:
                    continue
                if rank[left] < rank[right]:
                    wins[index[left], index[right]] += 1

    return wins


def condorcet_winner(profile, candidates):
    wins = pairwise_matrix(profile, candidates)
    winners = []
    for i, candidate in enumerate(candidates):
        row = wins[i]
        column = wins[:, i]
        beats_all = np.all(row[np.arange(len(candidates)) != i] > column[np.arange(len(candidates)) != i])
        if beats_all:
            winners.append(candidate)

    return tuple(winners)


def majority_winner(left, right, wins, candidates, tie_break):
    index = {candidate: i for i, candidate in enumerate(candidates)}
    left_votes = wins[index[left], index[right]]
    right_votes = wins[index[right], index[left]]
    if left_votes > right_votes:
        return left
    if right_votes > left_votes:
        return right
    for candidate in tie_break:
        if candidate in (left, right):
            return candidate
    raise ValueError("tie_break must contain both candidates")


def agenda_winner(profile, candidates, agenda, tie_break=None):
    if tie_break is None:
        tie_break = candidates
    wins = pairwise_matrix(profile, candidates)
    current = majority_winner(agenda[0], agenda[1], wins, candidates, tie_break)
    for challenger in agenda[2:]:
        current = majority_winner(current, challenger, wins, candidates, tie_break)

    return current


cycle_profile = [
    ("A", "B", "C"),
    ("B", "C", "A"),
    ("C", "A", "B"),
]
cycle_wins = pairwise_matrix(cycle_profile, lesson_candidates)

assert cycle_wins[0, 1] == 2
assert cycle_wins[1, 0] == 1
assert cycle_wins[1, 2] == 2
assert cycle_wins[2, 1] == 1
assert cycle_wins[2, 0] == 2
assert cycle_wins[0, 2] == 1
assert condorcet_winner(cycle_profile, lesson_candidates) == tuple()
assert agenda_winner(cycle_profile, lesson_candidates, ("A", "B", "C")) == "C"
assert agenda_winner(cycle_profile, lesson_candidates, ("B", "C", "A")) == "A"
assert agenda_winner(cycle_profile, lesson_candidates, ("C", "A", "B")) == "B"

print("Pairwise majority matrix rows beat columns:")
print(cycle_wins)
print("Agenda winners:", "C", "A", "B")


## Dataset ladder: D1–D5 profiles of rising size and conflict

The ladder starts with the three-voter Condorcet cycle, then adds the five-ballot plurality-versus-Borda lesson profile, agenda variants, tie-heavy larger elections, and a strategic-manipulation profile where rule, agenda, and tie-break all matter.

In [ ]:
def build_voting_ladder():
    d1 = {
        "name": "D1 Condorcet cycle",
        "profile": cycle_profile,
        "candidates": lesson_candidates,
        "agenda": ("A", "B", "C"),
        "tie_break": lesson_candidates,
    }
    d2 = {
        "name": "D2 plurality tie, Borda winner",
        "profile": lesson_profile,
        "candidates": lesson_candidates,
        "agenda": ("A", "B", "C"),
        "tie_break": lesson_candidates,
    }
    d3 = {
        "name": "D3 agenda-dependent cycle",
        "profile": cycle_profile + [("A", "C", "B"), ("B", "A", "C"), ("C", "B", "A")],
        "candidates": lesson_candidates,
        "agenda": ("C", "A", "B"),
        "tie_break": ("B", "A", "C"),
    }
    d4_candidates = ("A", "B", "C", "D")
    d4_profile = [
        ("A", "B", "C", "D"),
        ("A", "C", "D", "B"),
        ("B", "C", "D", "A"),
        ("B", "D", "A", "C"),
        ("C", "D", "A", "B"),
        ("C", "A", "B", "D"),
        ("D", "A", "B", "C"),
        ("D", "B", "C", "A"),
    ]
    d4 = {
        "name": "D4 four-candidate tie-heavy survey",
        "profile": d4_profile,
        "candidates": d4_candidates,
        "agenda": d4_candidates,
        "tie_break": d4_candidates,
    }
    d5_candidates = ("A", "B", "C", "D", "E")
    d5_profile = [
        ("A", "B", "C", "D", "E"),
        ("A", "C", "B", "E", "D"),
        ("B", "C", "D", "E", "A"),
        ("B", "D", "E", "A", "C"),
        ("C", "D", "E", "A", "B"),
        ("C", "E", "A", "B", "D"),
        ("D", "E", "A", "B", "C"),
        ("D", "A", "B", "C", "E"),
        ("E", "A", "B", "C", "D"),
        ("E", "B", "C", "D", "A"),
        ("A", "D", "B", "C", "E"),
        ("B", "E", "C", "D", "A"),
    ]
    d5 = {
        "name": "D5 strategic manipulation and tie-breaking",
        "profile": d5_profile,
        "candidates": d5_candidates,
        "agenda": ("A", "B", "C", "D", "E"),
        "tie_break": ("B", "A", "C", "D", "E"),
    }
    return [d1, d2, d3, d4, d5]


voting_ladder = build_voting_ladder()
for rung in voting_ladder:
    profile = rung["profile"]
    candidates = rung["candidates"]
    print(rung["name"])
    print("  voters:", len(profile), "candidates:", len(candidates))
    print("  first two ballots:", profile[:2])


In [ ]:
voting_results = []
for rung in voting_ladder:
    profile = rung["profile"]
    candidates = rung["candidates"]
    tie_break = rung["tie_break"]
    plural_rule_winner, plural_counts = plurality(profile, candidates, tie_break)
    borda_rule_winner, borda_scores = borda(profile, candidates)
    agenda_rule_winner = agenda_winner(profile, candidates, rung["agenda"], tie_break)
    condorcet = condorcet_winner(profile, candidates)
    welfare = max(borda_scores.values())
    stability = len(set([plural_rule_winner, borda_rule_winner[0], agenda_rule_winner]))
    voting_results.append({
        "name": rung["name"],
        "plurality": plural_rule_winner,
        "borda": borda_rule_winner[0],
        "agenda": agenda_rule_winner,
        "condorcet": condorcet,
        "welfare": welfare,
        "stability": stability,
        "counts": plural_counts,
        "scores": borda_scores,
        "wins": pairwise_matrix(profile, candidates),
    })

print("rung | plurality | borda | agenda | condorcet | max Borda welfare | distinct rule winners")
for result in voting_results:
    print(result["name"], result["plurality"], result["borda"], result["agenda"], result["condorcet"], result["welfare"], result["stability"])


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for col, result in enumerate(voting_results):
    ax_profile = axes[0, col]
    candidates = voting_ladder[col]["candidates"]
    counts = [result["scores"][candidate] for candidate in candidates]
    ax_profile.bar(candidates, counts, color="steelblue")
    ax_profile.set_title(result["name"].split()[0] + " Borda")
    ax_profile.set_ylim(0, max(counts) + 3)

    ax_pair = axes[1, col]
    image = ax_pair.imshow(result["wins"], cmap="RdBu", vmin=0)
    ax_pair.set_xticks(range(len(candidates)))
    ax_pair.set_xticklabels(candidates)
    ax_pair.set_yticks(range(len(candidates)))
    ax_pair.set_yticklabels(candidates)
    ax_pair.set_title("pairwise votes")
    for i in range(len(candidates)):
        for j in range(len(candidates)):
            ax_pair.text(j, i, str(result["wins"][i, j]), ha="center", va="center", fontsize=8)

fig.colorbar(image, ax=axes[:, -1], shrink=0.7)
fig.tight_layout()
plt.show()

sizes = [len(rung["profile"]) * len(rung["candidates"]) for rung in voting_ladder]
welfare = [result["welfare"] for result in voting_results]
stability = [result["stability"] for result in voting_results]

plt.figure(figsize=(7, 4))
plt.plot(sizes, welfare, marker="o", label="max Borda welfare")
plt.plot(sizes, stability, marker="s", label="distinct rule winners")
plt.xlabel("profile size: voters × candidates")
plt.ylabel("metric")
plt.title("Outcome vs. profile size/conflict")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Pitfall on D5: Condorcet cycles and strategic voting

The wrong audit says a winner is obvious after one plurality count. The fix is to name the rule, tie-break, and agenda, then report whether a strategic ballot can change the declared outcome.

In [ ]:
d5 = voting_ladder[-1]
truthful_profile = d5["profile"]
wrong_winner, wrong_counts = plurality(truthful_profile, d5["candidates"])
strategic_profile = list(truthful_profile)
strategic_profile[0] = ("B", "A", "C", "D", "E")
fixed_winner, fixed_counts = plurality(strategic_profile, d5["candidates"], d5["tie_break"])
truthful_agenda = agenda_winner(truthful_profile, d5["candidates"], d5["agenda"], d5["tie_break"])
strategic_agenda = agenda_winner(strategic_profile, d5["candidates"], d5["agenda"], d5["tie_break"])

print("Wrong plurality-only winner set:", wrong_winner)
print("Truthful plurality counts:", wrong_counts)
print("Strategic plurality counts:", fixed_counts)
print("Declared plurality winner with tie-break:", fixed_winner)
print("Agenda winner before strategic ballot:", truthful_agenda)
print("Agenda winner after strategic ballot:", strategic_agenda)
print("Fix: publish the rule, tie-break, agenda, Borda welfare, and Condorcet status together.")


## Evaluate it + Practice

- Metric: report the winner under plurality, Borda, agenda majority, and the max Borda welfare score; compare against a no-skill baseline that always picks the first candidate by tie-break.
- Cheap sanity check: pairwise matrices must satisfy $M_{ij}+M_{ji}=n$ for every candidate pair.
- Ablation: remove lower-rank information by switching Borda to plurality and watch D2 lose its unique Borda winner.
- Failure signals: a Condorcet cycle reported as a Condorcet winner, an undeclared tie-break, or an agenda winner shown without the agenda.

Practice prompt 1: Change the D3 agenda and predict the new final winner before running the cell below.

Practice prompt 2: Add one candidate to D4 and compare plurality, Borda, and pairwise-majority outcomes.

Practice prompt 3: Construct a ballot change that helps a preferred candidate under plurality but hurts them under Borda.